In [1]:
# Parameters
city_key = "ywg"
baseline_feed_id = "avg_pre_ptn"
comparison_feed_id = "current"
save_figures = False
figures_dir = "reports/pr2/figures"
dpi = 200

# PR2: Final Figure Package

This notebook assembles Stephenie's PR2 visual layer from canonical shared outputs. It is the last-pass packaging notebook for the report and dashboard, not a separate analysis path.

## 1. Setup & Helpers

In [2]:
from pathlib import Path
from shapely import wkt
import warnings

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

from ptn_analysis.context.reporting import (
    ensure_report_dirs,
    save_placeholder_figure,
    save_report_figure,
)
from ptn_analysis.context.reporting import collect_summary_stats
from ptn_analysis.context.config import (
    PTN_TIER_COLORS,
    PTN_TIER_ORDER,
    SERVING_DUCKDB_PATH,
)
from ptn_analysis.context.db import TransitDB

ensure_report_dirs("pr2")
figure_output_directory = Path(figures_dir)
figure_output_directory.mkdir(parents=True, exist_ok=True)

serving_db = TransitDB(SERVING_DUCKDB_PATH)

## 2. Dashboard KPI Table

In [3]:
# KPI Summary Table — Stephenie

try:
    kpi = collect_summary_stats(
        db_instance=serving_db,
        city_key=city_key,
        feed_id=comparison_feed_id,
    )
    print("KPI raw dict:", kpi)
except Exception as e:
    print("KPI collection failed:", repr(e))
    kpi = {
        "num_stops": None,
        "num_edges": None,
        "neighbourhood_count": None,
        "route_count": None,
        "total_neighbourhood_stops": None,
        "min_stop_density_per_km2": None,
        "median_stop_density_per_km2": None,
        "max_stop_density_per_km2": None,
        "jobs_access_neighbourhood_count": None,
        "mean_jobs_access_score": None,
        "max_jobs_access_score": None,
    }

kpi_table = pd.DataFrame(
    {"Metric": list(kpi.keys()), "Value": list(kpi.values())}
)

kpi_table

2026-03-12 13:47:19.170 | INFO     | ptn_analysis.context.db:engine:62 - SQLAlchemy engine created for /Users/stephanie/ywg-ptn-analysis-4710-g11/data/processed/wpg_transit_serving.duckdb


KPI raw dict: {'num_stops': 3873, 'num_edges': 4663, 'neighbourhood_count': 237, 'route_count': 71, 'total_neighbourhood_stops': 3873, 'min_stop_density_per_km2': 0.0, 'median_stop_density_per_km2': 11.722853382315575, 'max_stop_density_per_km2': 70.80869084890885, 'jobs_access_neighbourhood_count': 237, 'mean_jobs_access_score': 90.12713966244726, 'max_jobs_access_score': 625.7024}


,Metric,Value
0,num_stops,3873.000000
1,num_edges,4663.000000
2,neighbourhood_count,237.000000
3,route_count,71.000000
4,total_neighbourhood_stops,3873.000000
5,min_stop_density_per_km2,0.000000
6,median_stop_density_per_km2,11.722853
7,max_stop_density_per_km2,70.808691
8,jobs_access_neighbourhood_count,237.000000
9,mean_jobs_access_score,90.127140


## 3. PR2 Summary Panel

In [4]:
# Figure 1: PR2 Summary Panel — Stephenie

tables = serving_db.query("SHOW TABLES")["name"].tolist()

metrics = serving_db.query("""
    SELECT *
    FROM ywg_route_schedule_metrics
    WHERE feed_id = :fid
""", {"fid": comparison_feed_id})

tiers = None

if "ywg_route_ptn_tiers" in tables:
    tiers = serving_db.query("""
        SELECT feed_id, route_id, ptn_tier
        FROM ywg_route_ptn_tiers
        WHERE feed_id = :fid
    """, {"fid": comparison_feed_id})

elif "ywg_route_classification_features" in tables:
    cols = serving_db.query(
        "PRAGMA table_info('ywg_route_classification_features')"
    )["name"].tolist()

    tier_col = next((c for c in ["ptn_tier", "tier_label", "route_tier"] if c in cols), None)
    has_feed = "feed_id" in cols

    if tier_col is not None:
        if has_feed:
            tiers = serving_db.query(f"""
                SELECT feed_id, route_id, {tier_col} AS ptn_tier
                FROM ywg_route_classification_features
                WHERE feed_id = :fid
            """, {"fid": comparison_feed_id})
        else:
            tiers = serving_db.query(f"""
                SELECT route_id, {tier_col} AS ptn_tier
                FROM ywg_route_classification_features
            """)

if tiers is not None and not tiers.empty:
    if "feed_id" in tiers.columns and "feed_id" in metrics.columns:
        tier_stats = metrics.merge(tiers, on=["feed_id", "route_id"], how="left")
    else:
        tier_stats = metrics.merge(tiers, on="route_id", how="left")
else:
    tier_stats = metrics.copy()
    tier_stats["ptn_tier"] = "Unknown"

tier_stats["ptn_tier"] = tier_stats["ptn_tier"].fillna("Unknown")

summary = (
    tier_stats.groupby("ptn_tier", dropna=False)
    .agg(
        avg_headway=("mean_headway_minutes", "mean"),
        avg_speed=("scheduled_speed_kmh", "mean"),
        route_count=("route_id", "nunique"),
    )
    .reset_index()
)

summary["sort_order"] = summary["ptn_tier"].apply(
    lambda x: PTN_TIER_ORDER.index(x) if x in PTN_TIER_ORDER else 999
)
summary = summary.sort_values("sort_order")

colors = [PTN_TIER_COLORS.get(t, "#7a7a7a") for t in summary["ptn_tier"]]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.patch.set_facecolor("white")

# Average headway
axes[0, 0].bar(summary["ptn_tier"], summary["avg_headway"], color=colors, edgecolor="black", linewidth=0.6)
axes[0, 0].set_title("Average Headway (min)", fontsize=13, weight="bold")
axes[0, 0].tick_params(axis="x", rotation=40)
axes[0, 0].grid(axis="y", alpha=0.25)

# Average speed
axes[0, 1].bar(summary["ptn_tier"], summary["avg_speed"], color=colors, edgecolor="black", linewidth=0.6)
axes[0, 1].set_title("Scheduled Speed (km/h)", fontsize=13, weight="bold")
axes[0, 1].tick_params(axis="x", rotation=40)
axes[0, 1].grid(axis="y", alpha=0.25)

# Route count
axes[1, 0].bar(summary["ptn_tier"], summary["route_count"], color=colors, edgecolor="black", linewidth=0.6)
axes[1, 0].set_title("Route Count", fontsize=13, weight="bold")
axes[1, 0].tick_params(axis="x", rotation=40)
axes[1, 0].grid(axis="y", alpha=0.25)

# KPI text
axes[1, 1].axis("off")
axes[1, 1].text(
    0.02,
    0.95,
    "\n".join([
        f"Stops: {kpi.get('num_stops', 'NA')}",
        f"Routes: {kpi.get('route_count', 'NA')}",
        f"Neighbourhoods: {kpi.get('neighbourhood_count', 'NA')}",
        f"Median stop density: {kpi.get('median_stop_density_per_km2', 'NA')}",
    ]),
    va="top",
    ha="left",
    fontsize=13,
    bbox=dict(boxstyle="round,pad=0.5", facecolor="#f5f5f5", edgecolor="#d0d0d0")
)

fig.suptitle("PTN Service Summary by Tier", fontsize=18, weight="bold")
plt.tight_layout()

save_report_figure(
    fig,
    "pr2_summary_panel.png",
    figures_dir=figure_output_directory,
    dpi=dpi,
    enabled=save_figures,
)

plt.close(fig)

## 4. Coverage Cluster Map

In [5]:
# Figure 2: Coverage Cluster Map — Stephenie
# Clean geometry-free version for reliability

density_df = serving_db.query("""
    SELECT neighbourhood, stop_density_per_km2
    FROM ywg_neighbourhood_stop_count_density
    WHERE feed_id = :fid
""", {"fid": comparison_feed_id})

density_df["coverage_cat"] = density_df["stop_density_per_km2"].apply(
    lambda d: "High" if pd.notna(d) and d >= 5 else (
        "Medium" if pd.notna(d) and d >= 1 else "Low"
    )
)

cat_counts = (
    density_df.groupby("coverage_cat")
    .size()
    .reindex(["High", "Medium", "Low"], fill_value=0)
    .reset_index(name="count")
)

cat_colors = {
    "High": "#1a9850",
    "Medium": "#fee08b",
    "Low": "#d73027",
}

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("white")

bars = ax.bar(
    cat_counts["coverage_cat"],
    cat_counts["count"],
    color=[cat_colors[c] for c in cat_counts["coverage_cat"]],
    edgecolor="black",
    linewidth=0.7,
)

ax.set_title("Neighbourhood Coverage Categories", fontsize=18, weight="bold")
ax.set_xlabel("Coverage Category", fontsize=12)
ax.set_ylabel("Number of Neighbourhoods", fontsize=12)
ax.grid(axis="y", alpha=0.25)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1, f"{int(h)}", ha="center", fontsize=12)

plt.tight_layout()

save_report_figure(
    fig,
    "coverage_cluster_map.png",
    figures_dir=figure_output_directory,
    dpi=dpi,
    enabled=save_figures,
)

plt.close(fig)

## 5. Network Community Map

In [6]:
# Figure 3: Network Community Map — Stephenie

community_df = serving_db.query("""
    SELECT stop_id, community_id
    FROM ywg_network_communities
""")

stops_df = serving_db.query("""
    SELECT stop_id, stop_lat, stop_lon
    FROM ywg_stops
""")

df = stops_df.merge(community_df, on="stop_id", how="inner")

fig, ax = plt.subplots(figsize=(10, 10))
fig.patch.set_facecolor("white")
ax.set_facecolor("#f7f7f7")

scatter = ax.scatter(
    df["stop_lon"],
    df["stop_lat"],
    c=df["community_id"],
    cmap="tab20",
    s=10,
    alpha=0.85,
    linewidths=0,
)

ax.set_title("Transit Network Communities", fontsize=18, weight="bold")
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)
ax.grid(alpha=0.15)

plt.tight_layout()

save_report_figure(
    fig,
    "network_communities_pr2.png",
    figures_dir=figure_output_directory,
    dpi=dpi,
    enabled=save_figures,
)

plt.close(fig)

## 5b. Before/After Route Map

In [ ]:
# Figure 4: Before/After PTN Route Comparison — Stephenie
# Keep filename: before_after_routes.png

from pathlib import Path
import duckdb

interim_path = Path("data/interim/wpg_transit.duckdb")
con = duckdb.connect(str(interim_path), read_only=True)

# Find a route table in the interim DB that has both feed_id and route_short_name
tables_df = con.execute("SHOW TABLES").fetchdf()
table_names = tables_df.iloc[:, 0].astype(str).tolist()

route_table = None
for tbl in table_names:
    try:
        cols = con.execute(f"DESCRIBE {tbl}").fetchdf()["column_name"].astype(str).tolist()
    except Exception:
        continue
    if "feed_id" in cols and "route_short_name" in cols:
        route_table = tbl
        break

if route_table is None:
    raise ValueError(
        f"Could not find a route table with feed_id and route_short_name in interim DB. "
        f"Available tables sample: {table_names[:20]}"
    )

print("Using interim route table:", route_table)

feeds_df = con.execute(f"""
    SELECT DISTINCT CAST(feed_id AS VARCHAR) AS feed_id
    FROM {route_table}
    ORDER BY 1
""").fetchdf()

available_feeds = feeds_df["feed_id"].tolist()
print("Available feeds:", available_feeds)

comparison_feed = str(comparison_feed_id)

# Choose an explicit pre-PTN baseline if present
preferred_pre_ptn = ["2025-04-13", "2024-12-15", "2024-09-01", "2024-06-16"]
baseline_feed = next((f for f in preferred_pre_ptn if f in available_feeds), None)

if baseline_feed is None:
    # fallback: pick the last non-current feed if available
    non_current = [f for f in available_feeds if f != comparison_feed]
    if not non_current:
        raise ValueError(f"No baseline feed found. Available feeds: {available_feeds}")
    baseline_feed = non_current[-1]

print("Baseline feed:", baseline_feed)
print("Comparison feed:", comparison_feed)

pre_routes = con.execute(f"""
    SELECT DISTINCT TRIM(CAST(route_short_name AS VARCHAR)) AS route_short_name
    FROM {route_table}
    WHERE CAST(feed_id AS VARCHAR) = ?
""", [baseline_feed]).fetchdf()

post_routes = con.execute(f"""
    SELECT DISTINCT TRIM(CAST(route_short_name AS VARCHAR)) AS route_short_name
    FROM {route_table}
    WHERE CAST(feed_id AS VARCHAR) = ?
""", [comparison_feed]).fetchdf()

pre_set = set(pre_routes["route_short_name"].dropna().astype(str))
post_set = set(post_routes["route_short_name"].dropna().astype(str))

retained_routes = sorted(pre_set & post_set)
gained_routes = sorted(post_set - pre_set)
lost_routes = sorted(pre_set - post_set)

retained = len(retained_routes)
gained = len(gained_routes)
lost = len(lost_routes)

print("Retained sample:", retained_routes[:10])
print("Gained sample:", gained_routes[:10])
print("Lost sample:", lost_routes[:10])

comp_df = pd.DataFrame({
    "Category": ["Retained", "Gained", "Lost"],
    "Count": [retained, gained, lost],
})

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor("white")

colors = ["#4daf4a", "#377eb8", "#e41a1c"]

bars = ax.bar(
    comp_df["Category"],
    comp_df["Count"],
    color=colors,
    edgecolor="black",
    linewidth=0.7,
)

ax.set_title("Before/After PTN Route Comparison", fontsize=18, weight="bold")
ax.set_xlabel("Category", fontsize=12)
ax.set_ylabel("Number of Routes", fontsize=12)
ax.grid(axis="y", alpha=0.25)

for bar in bars:
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        h + 0.3,
        f"{int(h)}",
        ha="center",
        fontsize=12
    )

subtitle = f"Baseline: {baseline_feed}   |   Comparison: {comparison_feed}"
ax.text(
    0.5, 1.00, subtitle,
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=10,
    color="dimgray"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])

save_report_figure(
    fig,
    "before_after_routes.png",
    figures_dir=figure_output_directory,
    dpi=dpi,
    enabled=save_figures,
)

plt.close(fig)
con.close()

## 6. QA Notes

Stephenie should verify that every exported filename matches `ptn_analysis/reporting.py`, that no notebook uses a local palette outside the shared visual helpers, and that the dashboard shows the same metrics as the report tables.